In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import pickle
from sklearn.preprocessing import OneHotEncoder
import torch
import torch.nn as nn
from tqdm import tqdm 
import sys
from mlp_classifier import MLPClassifier, train_mlp, predict_mlp
from scm import SCM, get_followups
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
from collections import defaultdict



In [2]:
lab_thresholds = {
    'CHOL-C': ('<', 5.2),
    'HDL-C': ('>', 1.2),
    'LDL-C_D': ('<', 3.4),
    'GLU-L': ('-', 3.6, 6.0),
    'HbA1C': ('-', 4.2, 6.1),
    'GPT-L': ('<', 40),
    'GGT-L': ('-', 7, 50),
    'GOT-L': ('<', 40),
    'KREA-L': ('-', 62, 106),
    'EGFR': ('<', 60),
    'CRP-L': ('<', 5.2)
}

diseases = {
    'hepatopathia': ['k70', 'k71', 'k72', 'k73', 'k74', 'k75', 'k76', 'k77'],
    'zsírmáj':  ['k760'],
    'diabetes': ['e10', 'e11', 'e12', 'e13', 'e14'],
    'cardiovascular disease': ['i20', 'i21', 'i22', 'i23', 'i24', 'i25'],
    'cererovascular disease': ['i63', 'i64', 'i65', 'i66', 'i67', 'i68'],
    'periferial arterial disease':  ['i73', 'i74'],
    'thrombosis': ['i80', 'i81', 'i82'],
    'renal diseases': ['n03', 'n11', 'n18'],
    'thyeroid diseases': ['e00', 'e01', 'e02', 'e03', 'e04', 'e05', 'e06', 'e07'],
    'atherosclerosis': ['i70'],
    'lipid metabolism disorders':  ['e78']
}
rename_diseases = {
    'hepatopathia': 'Hepatopathia',
    'zsírmáj': 'Fatty liver',
    'diabetes': 'Diabetes',
    'cardiovascular disease': 'Cardiovascular Disease',
    'cererovascular disease': 'Cererovascular Disease',
    'periferial arterial disease': 'Periferial Arterial Disease',
    'thrombosis': 'Thrombosis',
    'renal diseases': 'Renal Diseases',
    'thyeroid diseases': 'Thyeroid Diseases',
    'atherosclerosis': 'Atherosclerosis',
    'lipid metabolism disorders': 'Lipid Metabolism Disorders'
}

diseases_to_GBD_names = {
        'hepatopathia': 'Cirrhosis and other chronic liver diseases	',
    'zsírmáj':  'Nonalcoholic fatty liver disease including cirrhosis',
    'diabetes': 'Diabetes mellitus',
    'cardiovascular disease': 'Ischemic heart disease',
    'cererovascular disease': 'Stroke',
    'periferial arterial disease':  'Lower extremity peripheral arterial disease',
    'thrombosis': 'No good match',
    'renal diseases': 'Chronic kidney disease',
    'thyeroid diseases': 'Thyroid diseases',
    'atherosclerosis': 'No good match',
    'lipid metabolism disorders':  'No good match'
}

labs = list(lab_thresholds.keys())

observation_cols = labs + ['PRESSURE_CAT', 'BMI_CAT', 'ESETKOR', 'NEM']
treatment_col = 'TIPUS'
diag_cols = list(rename_diseases.values())

device = 'cuda:0'

In [3]:
gbd_df = pd.read_csv('GBD/IHME-GBD_2023_DATA-ad0f6a03-1.csv')
gbd_df['val'] = gbd_df['val'] / 100000
gbd_df = gbd_df[gbd_df['sex_name'] != 'Both']

sex_age_disease_to_GBDs = {}
for sex_name in gbd_df['sex_name'].unique():
    sex = 'M' if sex_name == 'Male' else 'F'
    sex_age_disease_to_GBDs[sex] = {}
    for age in gbd_df['age_name'].unique():
        daly_list = []
        for disease in diseases.keys():
            if diseases_to_GBD_names[disease] in gbd_df['cause_name'].unique():
                daly_list.append(gbd_df[(gbd_df['sex_name'] == sex_name) & (gbd_df['age_name'] == age) & (gbd_df['cause_name'] == diseases_to_GBD_names[disease])]['val'].to_numpy()[0])
            else:
                daly_list.append(0)
        if age == '<1 year':
            low = 0
            high = 1
        elif age == '95+ years':
            low = 95
            high = 120
        else:
            low = int(age.split('-')[0])
            high = int(age.split('-')[1].split(' years')[0])
        for age_i in range(low, high+1):
            sex_age_disease_to_GBDs[sex][age_i] = np.array(daly_list)


In [4]:
sglt_processed = pd.read_csv('../data/sglt2_processed_with_lab_cats_and_textual_diags_with_missing.csv')
sglt_old_processed = pd.read_csv('../data/sglt2_old_processed_with_lab_cats_and_multiple_diags_with_missing.csv')
control_2_processed = pd.read_csv('../data/sglt2_control_2_processed_with_lab_cats_and_multiple_diags_with_missing.csv')
sglt_old_processed['TIPUS'] = 'No treatment'
control_2_processed['TIPUS'] = 'No treatment'

# rename columns in sglt_processed
sglt_processed = sglt_processed.rename(columns={col: col.replace('_x', '') for col in sglt_processed.columns if '_x' in col})
sglt_processed = sglt_processed.drop(columns=[col for col in sglt_processed.columns if '_y' in col])

sglt_full = pd.concat([sglt_processed, sglt_old_processed, control_2_processed], ignore_index=True) # Use this in final
# sglt_full = pd.concat([sglt_processed, sglt_old_processed], ignore_index=True)


for disease in diseases.keys():
    sglt_full[disease]  = sglt_full[disease].replace(['nan', 'None', ''], None)
    sglt_full[disease] = sglt_full[disease].astype('category')

for obs in observation_cols:
    sglt_full[obs] = sglt_full[obs].apply(lambda x: str(x)).replace(['nan', 'None', ''], None)
    sglt_full[obs] = sglt_full[obs].astype('category')


sglt_full['ESETKOR'] = sglt_full['ESETKOR'].apply(lambda x: str(x)).replace(['nan', 'None', ''], None)
sglt_full['ESETKOR'] = sglt_full['ESETKOR'].astype(float)
age_cats = sglt_full['ESETKOR'].quantile([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]).to_dict()
sglt_full['ESETKOR'] = pd.cut(sglt_full['ESETKOR'], bins=[-1] + list(age_cats.values()), labels=list(age_cats.keys()), include_lowest=True)
sglt_full['ESETKOR'] = sglt_full['ESETKOR'].astype('category')

sglt_full['TIPUS'] = sglt_full['TIPUS'].apply(lambda x: str(x))
sglt_full['TIPUS'] = sglt_full['TIPUS'].astype('category')

sglt_full = sglt_full.rename(columns=rename_diseases)

sglt_full = sglt_full.dropna(subset=observation_cols, how='all')
sglt_full = sglt_full.dropna(subset=diag_cols, how='all')
sglt_full = sglt_full.dropna(subset=[treatment_col], how='all')

sglt_full = sglt_full.drop_duplicates(subset=['ESETAZON'], keep='first')

sglt_full = sglt_full.sort_values(by=['SZEMELYID', 'ESETAZON'])
sglt_full['Next ESETAZON'] = sglt_full.groupby('SZEMELYID')['ESETAZON'].shift(-1).astype('Int64')

# split based on szemelyid
train_szemelyid, test_szemelyid = train_test_split(sglt_full['SZEMELYID'].unique(), test_size=0.1, random_state=42, shuffle=True)

DR_ratio = 0.4

scm_train_szemelyid, DR_train_szemelyid = train_test_split(train_szemelyid, test_size=DR_ratio, random_state=42, shuffle=True)
scm_train_szemelyid, policy_train_szemelyid = train_test_split(scm_train_szemelyid, test_size=0.3, random_state=42, shuffle=True)

scm_test_szemelyid, DR_test_szemelyid = train_test_split(test_szemelyid, test_size=DR_ratio, random_state=42, shuffle=True)
scm_test_szemelyid, policy_test_szemelyid = train_test_split(scm_test_szemelyid, test_size=0.3, random_state=42, shuffle=True)

scm_train_X = sglt_full[sglt_full['SZEMELYID'].isin(scm_train_szemelyid)]
DR_train_X = sglt_full[sglt_full['SZEMELYID'].isin(DR_train_szemelyid)]
policy_train_X = sglt_full[sglt_full['SZEMELYID'].isin(policy_train_szemelyid)]

scm_test_X = sglt_full[sglt_full['SZEMELYID'].isin(scm_test_szemelyid)]
DR_test_X = sglt_full[sglt_full['SZEMELYID'].isin(DR_test_szemelyid)]
policy_test_X = sglt_full[sglt_full['SZEMELYID'].isin(policy_test_szemelyid)]

obs_variable_values = []
for obs in observation_cols:
    obs_variable_values.append(sorted(sglt_full[obs].dropna().unique().tolist()))

/tmp/ipykernel_543904/1621402506.py:1: DtypeWarning: Columns (26,37,39) have mixed types. Specify dtype option on import or set low_memory=False.
  sglt_processed = pd.read_csv('../data/sglt2_processed_with_lab_cats_and_textual_diags_with_missing.csv')
/tmp/ipykernel_543904/1621402506.py:3: DtypeWarning: Columns (22,26,28,31,33,34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  control_2_processed = pd.read_csv('../data/sglt2_control_2_processed_with_lab_cats_and_multiple_diags_with_missing.csv')


In [5]:
disease_means = scm_train_X[diag_cols].astype(float).mean()
disease_vars = scm_train_X[diag_cols].astype(float).var()

In [6]:
# with open('scm/sglt2_scm_model_balanced.pkl', 'rb') as f:
#     scm = pickle.load(f)

# is_no_treatment = scm_train_X['TIPUS'] == "No treatment"
# is_treatment = ~is_no_treatment
# scm_train_X = pd.concat([scm_train_X[is_treatment], scm_train_X[is_no_treatment].sample(n=len(scm_train_X[is_treatment])*2, random_state=1)], ignore_index=True)

scm = SCM(scm_train_X, obs_train_epochs=50, diag_train_epochs=50, obs_variable_values=obs_variable_values) 

# Uncomment to evaluate the scm

# # cases_with_followup = scm_test_X['Next ESETAZON'].notna() & scm_test_X['ESETAZON'].isin(scm_test_X['Next ESETAZON'])
# cases_with_followup = scm_test_X['Next ESETAZON'].notna() & scm_test_X['Next ESETAZON'].isin(scm_test_X['ESETAZON'])

# followups = get_followups(scm_test_X, scm_test_X.loc[cases_with_followup, 'Next ESETAZON'])
# obs_k = scm_test_X.loc[cases_with_followup, observation_cols]
# obs_k = scm.encode_data(obs_k, 'obs')
# diag_k = scm_test_X.loc[cases_with_followup, diag_cols]
# diag_k = torch.tensor(diag_k.astype(np.float32).to_numpy(), dtype=torch.float32, device=device)
# treatment_k = scm_test_X.loc[cases_with_followup, [treatment_col]]
# treatment_k = scm.encode_data(treatment_k, 'treatment')
# obs_k_next = followups.loc[:, observation_cols]

# scm.evaluate_obs(obs_k, diag_k, treatment_k, obs_k_next)

# obs_k = scm_test_X.loc[:, observation_cols]
# obs_k =  scm.encode_data(obs_k, 'obs', fit=False)
# diag_k = scm_test_X.loc[:, diag_cols]

# scm.evaluate_diag(obs_k, diag_k)

# save the scm model
# with open('scm/sglt2_scm_model_balanced.pkl', 'wb') as f:
#     pickle.dump(scm, f)

Fitting model for CHOL-C
Epoch [1/50], Loss: 0.6241
Epoch [2/50], Loss: 0.5436
Epoch [3/50], Loss: 0.5150
Epoch [4/50], Loss: 0.5128
Epoch [5/50], Loss: 0.5097
Epoch [6/50], Loss: 0.4966
Epoch [7/50], Loss: 0.4976
Epoch [8/50], Loss: 0.4771
Epoch [9/50], Loss: 0.4720
Epoch [10/50], Loss: 0.4534
Epoch [11/50], Loss: 0.4658
Epoch [12/50], Loss: 0.4516
Epoch [13/50], Loss: 0.4314
Epoch [14/50], Loss: 0.4411
Epoch [15/50], Loss: 0.4214
Epoch [16/50], Loss: 0.4329
Epoch [17/50], Loss: 0.4241
Epoch [18/50], Loss: 0.4146
Epoch [19/50], Loss: 0.4011
Epoch [20/50], Loss: 0.4064
Epoch [21/50], Loss: 0.3943
Epoch [22/50], Loss: 0.3843
Epoch [23/50], Loss: 0.3808
Epoch [24/50], Loss: 0.3668
Epoch [25/50], Loss: 0.3705
Epoch [26/50], Loss: 0.3507
Epoch [27/50], Loss: 0.3691
Epoch [28/50], Loss: 0.3785
Epoch [29/50], Loss: 0.3661
Epoch [30/50], Loss: 0.3735
Epoch [31/50], Loss: 0.3504
Epoch [32/50], Loss: 0.3650
Epoch [33/50], Loss: 0.3577
Epoch [34/50], Loss: 0.3475
Epoch [35/50], Loss: 0.3332
Epoc

In [7]:
from torch import threshold


def predict_diag_for_all_actions(data, indices, use_counterfactuals=False):
    diags = {}
    test_data = data.loc[indices].reset_index(drop=True)
    for t in sglt_full[treatment_col].cat.codes.unique():
        diag = np.zeros((len(test_data), len(diag_cols)), dtype=float)
        # diag = np.full((len(test_data), len(diag_cols)), np.nan, dtype=float)

        actions_in_data_indices = (test_data[treatment_col].cat.codes == t) & (test_data['Next ESETAZON'].isin(data['ESETAZON']).values)
        next_indices = test_data.loc[actions_in_data_indices, 'Next ESETAZON'].values.reshape(-1) 
        diag[actions_in_data_indices] = get_followups(data, next_indices).loc[:, diag_cols].astype(float).values

        if not actions_in_data_indices.all():
            n_missing = (~actions_in_data_indices).sum()
            scm_actions = pd.DataFrame(np.array([sglt_full[treatment_col].cat.categories[t] for _ in range(n_missing)]), columns=[treatment_col], index=indices.cpu().numpy()[~actions_in_data_indices])
            scm_actions = scm.encode_data(scm_actions, 'treatment', fit=False)

            scm_obs = test_data.loc[~actions_in_data_indices][observation_cols]
            scm_obs = scm.encode_data(scm_obs, 'obs', fit=False)

            scm_diags = torch.tensor(test_data.loc[~actions_in_data_indices, diag_cols].astype(np.float32).to_numpy(), dtype=torch.float32, device=device)

            ################ counterfactual_prediction
            contra_in_data = test_data.loc[:, 'Next ESETAZON'].isin(data['ESETAZON']).values & test_data.loc[:, 'Next ESETAZON'].notna().values & ~actions_in_data_indices
            contra_indices = get_followups(data, test_data.loc[contra_in_data, 'Next ESETAZON']).index

            contra_in_data = np.atleast_1d(contra_in_data).astype(bool)
            contra_in_data_action_not = contra_in_data[~actions_in_data_indices] # indexing for the already filtered arrays

            if contra_in_data.any():
                true_next_obs = data.set_index("ESETAZON").loc[contra_indices, observation_cols]
                scm_true_next_obs = scm.encode_data(true_next_obs, 'obs', fit=False)
                scm_next_obs_contra = scm.gumbel_sample_contra_obs(scm_obs[contra_in_data_action_not], scm_diags[contra_in_data_action_not], scm_actions[contra_in_data_action_not], scm_true_next_obs, nsamp=1000)
            #################

            scm_next_obs = scm.predict_obs(scm_obs, scm_diags, scm_actions)
            # insert conta obs
            if contra_in_data_action_not.any():
                scm_next_obs[contra_in_data_action_not] = scm_next_obs_contra
            scm_diag = scm.predict_diag(scm_next_obs, probs=True)
            diag[~actions_in_data_indices] = scm_diag.cpu().detach().numpy()

        diags[t] = diag
    
    return diags


def ITE(data, indices, T):
    diags = predict_diag_for_all_actions(data, indices, use_counterfactuals=True)

    ites = {}
    for t1 in diags.keys():
        ites[t1] = np.zeros((len(T), len(diag_cols)), dtype=float)
        for t2 in diags.keys():
            if t1 != t2:
                ites[t1] += diags[t2] - diags[t1] # set axis to see average over samples or features
        ites[t1] /= (len(diags.keys()) - 1)

    # choose according to T
    ite = np.zeros((len(T), len(diag_cols)), dtype=float)
    for t1 in ites.keys():
        ite[T==t1, :] = ites[t1][T==t1, :]

    return ite

def nash_social_welfare(improvements, epsilon=1e-9, axis=1):
    u_clipped = np.maximum(improvements, epsilon)
    return np.sum(np.log(u_clipped), axis=axis)

def alpha_fair_utility(improvements, alpha, axis=1):
    if alpha == 1:
        return torch.log(improvements + 1e-8).mean(axis=axis)
    else:
        return ((improvements ** (1 - alpha)) / (1 - alpha)).mean(axis=axis)
    
def soft_count_small(outcomes, threshold=0.5, k=20, axis=1):
    outcomes = np.asarray(outcomes)
    return np.sum(1 / (1 + np.exp(-k * (outcomes - threshold))), axis=axis)

def does_no_harm(outcomes, previous_diag):
    outcomes = np.asarray(outcomes)
    return np.all(outcomes <= previous_diag, axis=1)

def daly(outcomes, sexes, ages):
    dalys = []
    for (oc, s, a) in zip(outcomes, sexes, ages):
        dalys.append(sex_age_disease_to_GBDs[s][a] * oc)
    return np.array(dalys)


def best_action(data, indices, previous_diag=None, use_counterfactuals=False):
    diags = predict_diag_for_all_actions(data, indices, use_counterfactuals=use_counterfactuals)

    # if previous_diag is not None:
    #     diag_no_harm = {t: does_no_harm(diags[t], previous_diag) for t in diags}
    #     all_no_harm = np.vstack([diag_no_harm[t] for t in sorted(diags.keys())]).T
    #     noharm_exists = np.any(all_no_harm, axis=1)
    
    # Sum risks for each treatment
    diag_sums = {t: diags[t].sum(axis=1) for t in diags}
    all_sums = np.vstack([diag_sums[t] for t in sorted(diags.keys())]).T
    best_action = np.argmin(all_sums, axis=1)

    # diag_norm_sum = {t: ((diags[t] - disease_means.values) / np.sqrt(disease_vars.values)).sum(axis=1) for t in diags}
    # all_norm_sums = np.vstack([diag_norm_sum[t] for t in sorted(diags.keys())]).T
    # best_action = np.argmin(all_norm_sums, axis=1)

    # diag_medians = {t: np.median(diags[t], axis=1) for t in diags}  
    # all_medians = np.vstack([diag_medians[t] for t in sorted(diags.keys())]).T
    # best_action = np.argmin(all_medians, axis=1) # so far the best

    # diag_maxs = {t: np.max(diags[t], axis=1) for t in diags}
    # all_maxs = np.vstack([diag_maxs[t] for t in sorted(diags.keys())]).T
    # best_action = np.argmin(all_maxs, axis=1)

    # diag_mins = {t: np.min(diags[t], axis=1) for t in diags}
    # all_mins = np.vstack([diag_mins[t] for t in sorted(diags.keys())]).T
    # best_action = np.argmin(all_mins, axis=1)

    # diag_nash_sw = {t: nash_social_welfare(1 - diags[t], axis=1) for t in diags}
    # all_nash_sw = np.vstack([diag_nash_sw[t] for t in sorted(diags.keys())]).T
    # best_action = np.argmax(all_nash_sw, axis=1)

    # diag_alpha_fair_utility = {t: alpha_fair_utility(1 - torch.tensor(diags[t]), alpha=500, axis=1).cpu().numpy() for t in diags}
    # all_alpha_fair_utility = np.vstack([diag_alpha_fair_utility[t] for t in sorted(diags.keys())]).T
    # best_action = np.argmax(all_alpha_fair_utility, axis=1) 

    # diag_soft_counts = {t: soft_count_small(diags[t], threshold=0.5, k=20) for t in diags}
    # all_soft_counts = np.vstack([diag_soft_counts[t] for t in sorted(diags.keys())]).T
    # best_action = np.argmin(all_soft_counts, axis=1)

    # idx = indices.cpu().numpy() 
    # diag_dalys= {t: daly(diags[t], data.loc[idx, 'NEM'], [int(age_cats[a]) for a in data.loc[idx, 'ESETKOR']]).sum(axis=1) for t in diags}
    # all_dalys = np.vstack([diag_dalys[t] for t in sorted(diag_dalys.keys())]).T
    # best_action = np.argmin(all_dalys, axis=1)

    return best_action


In [8]:
DR_test_X = DR_test_X.reset_index(drop=True)

test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.arange(len(DR_test_X))),
    batch_size=32,
    shuffle=True
)

accuracy = 0
ate_true = 0
ate_policy = 0
ate_baseline = 0
ate_baseline_1 = 0
ate_baseline_2 = 0
ate_baseline_3 = 0
ate_best = 0
iters = len(test_loader)

observed_actions = np.array([], dtype=int)
policy_actions = np.array([], dtype=int)

real_diags = np.zeros((0, len(diag_cols)), dtype=float)
policy_diags = np.zeros((0, len(diag_cols)), dtype=float)
baseline_0_diags = np.zeros((0, len(diag_cols)), dtype=float)

x_obs_all = None
x_treatment_all = None
x_diag_all = None

for indices in tqdm(test_loader):
    
    indices = indices[0]
    x_obs = DR_test_X.iloc[indices][observation_cols]
    x_diag = DR_test_X.iloc[indices][diag_cols]
    x_action = DR_test_X.iloc[indices][treatment_col].cat.codes.values

    x_obs_all = pd.concat([x_obs_all, x_obs], ignore_index=True) if x_obs_all is not None else x_obs
    x_diag_all = pd.concat([x_diag_all, x_diag], ignore_index=True) if x_diag_all is not None else x_diag
    x_treatment_all = np.append(x_treatment_all, x_action) if x_treatment_all is not None else x_action

    actions = best_action(DR_test_X, indices, x_diag)

    accuracy += (actions == x_action).mean()

    ate_policy += ITE(DR_test_X, indices, actions).sum() / len(indices)
    ate_true += ITE(DR_test_X, indices, x_action).sum() / len(indices)
    ate_baseline += ITE(DR_test_X, indices, np.array([0 for _ in range(len(indices))])).sum() / len(indices)
    ate_baseline_1 += ITE(DR_test_X, indices, np.array([1 for _ in range(len(indices))])).sum() / len(indices)
    ate_baseline_2 += ITE(DR_test_X, indices, np.array([2 for _ in range(len(indices))])).sum() / len(indices)

    observed_actions = np.append(observed_actions, x_action)
    policy_actions = np.append(policy_actions, actions)

    all_diags = predict_diag_for_all_actions(DR_test_X, indices, use_counterfactuals=True)

    real_diags_i = np.zeros((len(indices), len(diag_cols)), dtype=float)
    policy_diags_i = np.zeros((len(indices), len(diag_cols)), dtype=float)

    for t in all_diags.keys():
        real_diags_i[x_action == t] = all_diags[t][x_action == t]
        policy_diags_i[actions == t] = all_diags[t][actions == t]
    

    real_diags = np.append(real_diags, real_diags_i, axis=0)
    policy_diags = np.append(policy_diags, policy_diags_i, axis=0)
    baseline_0_diags = np.append(baseline_0_diags, all_diags[0], axis=0)

print(f"Test accuracy: {accuracy / iters}")
print(f"ATE observed: {ate_true / iters}")
print(f"ATE policy: {ate_policy / iters}")
print(f"ATE difference: {(ate_policy - ate_true) / iters}")
print(f"ATE baseline: {ate_baseline / iters}")
print(f"ATE baseline difference: {(ate_policy - ate_baseline) / iters}")

print(f"ATE baseline 1: {ate_baseline_1 / iters}")
print(f"ATE baseline 2: {ate_baseline_2 / iters}")

100%|██████████| 62/62 [03:14<00:00,  3.13s/it]

Test accuracy: 0.6138104838709677
ATE observed: 0.4340472610673787
ATE policy: 0.9691010773750396
ATE difference: 0.5350538163076608
ATE baseline: 0.45482280414213405
ATE baseline difference: 0.5142782732329055
ATE baseline 1: -0.14047925489112012
ATE baseline 2: -0.1799929600619025


In [9]:
policy_train_X_obs = policy_train_X[observation_cols]
policy_train_X_obs = scm.encode_data(policy_train_X_obs, 'obs', fit=False) 
policy_train_X_diags = policy_train_X[diag_cols]
policy_train_X_diags = torch.tensor(policy_train_X_diags.astype(np.float32).to_numpy(), dtype=torch.float32, device=device)
scm_preds = {}
scm_best_actions = []
batch_size = 64
for start_idx in range(0, len(policy_train_X), batch_size):
    end_idx = min(start_idx + batch_size, len(policy_train_X))
    batch_indices = torch.tensor(policy_train_X.index[start_idx:end_idx])
    batch_scm_preds = predict_diag_for_all_actions(policy_train_X, batch_indices, use_counterfactuals=True)
    batch_best_actions = best_action(policy_train_X, batch_indices, use_counterfactuals=True)
    scm_best_actions.extend(batch_best_actions.tolist())
    for t in batch_scm_preds.keys():
        if t not in scm_preds:
            scm_preds[t] = batch_scm_preds[t]
        else:
            scm_preds[t] = np.vstack([scm_preds[t], batch_scm_preds[t]])

# scm_preds = predict_diag_for_all_actions(policy_train_X, torch.tensor(policy_train_X.index), use_counterfactuals=True)
scm_best_actions = np.array(scm_best_actions, dtype=int)
scm_preds_array = np.hstack([scm_preds[t] for t in sorted(scm_preds.keys())])
scm_outcomes = torch.tensor(scm_preds_array, dtype=torch.float).to(device)

# scm_best_actions = best_action(policy_train_X, torch.tensor(policy_train_X.index), use_counterfactuals=True)
scm_best_outcomes = torch.tensor(np.array([scm_preds[a][i] for i, a in enumerate(scm_best_actions)]), dtype=torch.float)


policy_data_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.cat([policy_train_X_obs.to(device), policy_train_X_diags.to(device)], dim=1), scm_outcomes.to(device), scm_best_outcomes.to(device), torch.arange(policy_train_X.shape[0]).to(device)),
    batch_size=16,
    shuffle=True,
)


class Q_network(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims=[128, 64, 32]):
        super(Q_network, self).__init__()
        layers = []
        prev_dim = input_dim
        for hdim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hdim))
            layers.append(nn.ReLU())
            prev_dim = hdim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

q_network = Q_network(policy_train_X_obs.shape[1] + policy_train_X_diags.shape[1], scm_outcomes.shape[1], hidden_dims=[512, 256, 128, 64, 32]).to(device)
# train_mlp(q_network, policy_data_loader, nn.MSELoss(), num_epochs=50)

def nash_social_welfare_torch(outcomes, epsilon=1e-9, dim=1):
    epsilons = torch.full_like(outcomes, epsilon)
    u_clipped = torch.maximum(outcomes, epsilons)
    return torch.sum(torch.log(u_clipped), dim=dim)

def daly_torch(outcomes, sexes, ages):
    dalys = []
    for (oc, s, a) in zip(outcomes, sexes, ages):
        dalys.append(torch.tensor(sex_age_disease_to_GBDs[s][a], device=device) * oc)
    return torch.stack(dalys)


def policy(x):
    decoded_data = scm.encoders['obs'].inverse_transform(x[:, :policy_train_X_obs.shape[1]].cpu().numpy())
    sexes = decoded_data[:, policy_train_X[observation_cols].columns.get_loc("NEM")] 
    ages = decoded_data[:, policy_train_X[observation_cols].columns.get_loc("ESETKOR")] # age
    ages = np.array([int(age_cats[a]) for a in ages])

    q_values = q_network(x)
    n_actions = len(sglt_full[treatment_col].cat.categories)
    action_selection = torch.stack([ torch.sum(q_values[:, i * q_values.shape[1] // n_actions : (i + 1) * q_values.shape[1] // n_actions], dim=1) for i in range(n_actions)]) # backup solution

    actions = torch.argmin(action_selection, dim=0).cpu().numpy() # sum median etc
    return actions

def best_outcomes(x, indices):
    actions = policy(x)
    outcomes = np.zeros((len(actions), scm_preds[0].shape[1]), dtype=np.float32)
    for a in np.unique(actions):
        a_indices = indices.cpu().numpy()[actions == a]
        outcomes[actions == a] = scm_preds[a][a_indices]
    return outcomes


q_network.train()
optimizer = torch.optim.Adam(q_network.parameters(), lr=1e-3, weight_decay=1e-5)
num_epochs = 20 # 100 

y_true_all = []
y_pred_all = []

regrets = []
for epoch in range(num_epochs):
    losses = []
    regret = 0
    for x_batch, y_batch, scm_best_outcomes_batch, indices_batch in policy_data_loader:
        logits = q_network(x_batch)
        loss = nn.MSELoss()(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        y_true_all.append(y_batch.detach().cpu())
        # regret += -(scm_best_outcomes_batch.cpu().numpy() - best_outcomes(x_batch, indices_batch)).sum().item()
        y_pred_all.append(logits.detach().cpu())

        
    regret /= len(policy_data_loader)
    regrets.append(regret)

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {np.mean(losses):.4f}, Regret: {regret:.4f}', flush=True)


Epoch [1/20], Loss: 0.0265, Regret: 0.0000
Epoch [2/20], Loss: 0.0211, Regret: 0.0000
Epoch [3/20], Loss: 0.0206, Regret: 0.0000
Epoch [4/20], Loss: 0.0202, Regret: 0.0000
Epoch [5/20], Loss: 0.0197, Regret: 0.0000
Epoch [6/20], Loss: 0.0194, Regret: 0.0000
Epoch [7/20], Loss: 0.0191, Regret: 0.0000
Epoch [8/20], Loss: 0.0188, Regret: 0.0000
Epoch [9/20], Loss: 0.0186, Regret: 0.0000
Epoch [10/20], Loss: 0.0183, Regret: 0.0000
Epoch [11/20], Loss: 0.0181, Regret: 0.0000
Epoch [12/20], Loss: 0.0179, Regret: 0.0000
Epoch [13/20], Loss: 0.0176, Regret: 0.0000
Epoch [14/20], Loss: 0.0174, Regret: 0.0000
Epoch [15/20], Loss: 0.0171, Regret: 0.0000
Epoch [16/20], Loss: 0.0169, Regret: 0.0000
Epoch [17/20], Loss: 0.0166, Regret: 0.0000
Epoch [18/20], Loss: 0.0164, Regret: 0.0000
Epoch [19/20], Loss: 0.0162, Regret: 0.0000
Epoch [20/20], Loss: 0.0159, Regret: 0.0000


In [10]:
import xgboost as xgb
from sklearn.multioutput import MultiOutputRegressor

# Prepare training data (same features as for neural network)
X_train_cf_gbt = torch.cat([policy_train_X_obs.cpu(), policy_train_X_diags.cpu()], dim=1).numpy()
y_train_cf_gbt = scm_outcomes.cpu().numpy()

# Counterfactual GBT with one model per action (trained on SCM counterfactual targets)
n_actions_gbt = len(sglt_full[treatment_col].cat.categories)
n_outcomes_per_action_cf = y_train_cf_gbt.shape[1] // n_actions_gbt

cf_train_sets = [
    (
        X_train_cf_gbt,
        y_train_cf_gbt[:, action_idx * n_outcomes_per_action_cf : (action_idx + 1) * n_outcomes_per_action_cf],
    )
    for action_idx in range(n_actions_gbt)
]

cf_gbt_models = [
    MultiOutputRegressor(
        xgb.XGBRegressor(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1
        )
    )
    for _ in range(n_actions_gbt)
]

for i, (X_train, y_train) in enumerate(cf_train_sets):
    cf_gbt_models[i].fit(X_train, y_train)

# Define counterfactual GBT-based policy
def gbt_policy(x):
    decoded_data = scm.encoders['obs'].inverse_transform(x[:, :policy_train_X_obs.shape[1]].cpu().numpy())
    sexes = decoded_data[:, policy_train_X[observation_cols].columns.get_loc("NEM")]
    ages = decoded_data[:, policy_train_X[observation_cols].columns.get_loc("ESETKOR")]
    ages = np.array([int(age_cats[a]) for a in ages])

    x_np = x.cpu().numpy() if isinstance(x, torch.Tensor) else x

    all_predictions = [m.predict(x_np) for m in cf_gbt_models]

    action_scores = []
    for action_outcomes in all_predictions:
        # action_score = daly(action_outcomes, sexes, ages).sum(axis=1)
        action_score = np.sum(action_outcomes, axis=1)
        action_scores.append(action_score)

    action_scores = np.stack(action_scores, axis=0)
    best_actions = np.argmin(action_scores, axis=0)
    return best_actions

In [11]:
cases_with_followup = policy_train_X['Next ESETAZON'].notna()

X_train_obs_gbt = torch.cat([policy_train_X_obs.cpu(), policy_train_X_diags.cpu()], dim=1).numpy()[cases_with_followup]
y_train_obs_gbt = get_followups(
    policy_train_X,
    policy_train_X[cases_with_followup]['Next ESETAZON']
).loc[:, diag_cols].astype(float).values

n_actions_obs_gbt = len(sglt_full[treatment_col].cat.categories)
# Each treatment-specific model predicts diagnosis outcomes only.
n_outcomes_per_action_obs = y_train_obs_gbt.shape[1]

treatment_indices_obs = policy_train_X[cases_with_followup][treatment_col].cat.codes.values
obs_train_sets = [
    (X_train_obs_gbt[treatment_indices_obs == a], y_train_obs_gbt[treatment_indices_obs == a])
    for a in range(n_actions_obs_gbt)
]

obs_gbt_models = [
    MultiOutputRegressor(xgb.XGBRegressor(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1
    )) for _ in range(n_actions_obs_gbt)
]

for i, (X_train, y_train) in enumerate(obs_train_sets):
    obs_gbt_models[i].fit(X_train, y_train)

# Define observational-only GBT-based policy
def obs_gbt_policy(x):
    decoded_data = scm.encoders['obs'].inverse_transform(x[:, :policy_train_X_obs.shape[1]].cpu().numpy())
    sexes = decoded_data[:, policy_train_X[observation_cols].columns.get_loc("NEM")]
    ages = decoded_data[:, policy_train_X[observation_cols].columns.get_loc("ESETKOR")]
    ages = np.array([int(age_cats[a]) for a in ages])

    x_np = x.cpu().numpy() if isinstance(x, torch.Tensor) else x

    all_predictions = [m.predict(x_np) for m in obs_gbt_models]
    all_predictions = np.concatenate(all_predictions, axis=1)

    action_scores = []
    for action_idx in range(n_actions_obs_gbt):
        start_idx = action_idx * n_outcomes_per_action_obs
        end_idx = (action_idx + 1) * n_outcomes_per_action_obs
        action_outcomes = all_predictions[:, start_idx:end_idx]

        # action_score = daly(action_outcomes, sexes, ages).sum(axis=1)
        # action_score = np.median(action_outcomes, axis=1)
        action_score = np.sum(action_outcomes, axis=1)
        action_scores.append(action_score)

    action_scores = np.stack(action_scores, axis=0)
    best_actions = np.argmin(action_scores, axis=0)
    return best_actions


In [12]:
# observational neural network trained only on observed follow-up data
cases_with_followup_obs_nn = policy_train_X['Next ESETAZON'].notna().values

X_train_obs_nn = torch.cat([policy_train_X_obs.cpu(), policy_train_X_diags.cpu()], dim=1)[cases_with_followup_obs_nn]
y_train_obs_nn = torch.tensor(
    get_followups(
        policy_train_X,
        policy_train_X.loc[cases_with_followup_obs_nn, 'Next ESETAZON']
    ).loc[:, diag_cols].astype(float).values,
    dtype=torch.float32,
 )

n_actions_obs_nn = len(sglt_full[treatment_col].cat.categories)
treatment_indices_obs_nn = policy_train_X.loc[cases_with_followup_obs_nn, treatment_col].cat.codes.values
obs_nn_train_counts = np.bincount(treatment_indices_obs_nn, minlength=n_actions_obs_nn)

class ObsOutcomeNetwork(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dims=[256, 128, 64]):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            prev_dim = hidden_dim
        layers.append(nn.Linear(prev_dim, output_dim))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

def train_obs_nn_model(model, data_loader, num_epochs=50, lr=1e-3):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    for epoch in range(num_epochs):
        model.train()
        epoch_losses = []
        for x_batch, y_batch in data_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_losses.append(loss.item())

    model.eval()
    return model

obs_nn_models = []
for action_idx in range(n_actions_obs_nn):
    action_mask = treatment_indices_obs_nn == action_idx
    if action_mask.sum() == 0:
        obs_nn_models.append(None)
        continue

    action_dataset = torch.utils.data.TensorDataset(
        X_train_obs_nn[action_mask],
        y_train_obs_nn[action_mask],
    )
    action_loader = torch.utils.data.DataLoader(
        action_dataset,
        batch_size=min(64, len(action_dataset)),
        shuffle=True,
    )

    action_model = ObsOutcomeNetwork(
        X_train_obs_nn.shape[1],
        y_train_obs_nn.shape[1],
    ).to(device)
    action_model = train_obs_nn_model(action_model, action_loader)
    obs_nn_models.append(action_model)

def obs_nn_predict_all_actions(x):
    x_tensor = x if isinstance(x, torch.Tensor) else torch.tensor(x, dtype=torch.float32)
    x_tensor = x_tensor.to(device=device, dtype=torch.float32)

    all_predictions = []
    for model in obs_nn_models:
        if model is None:
            all_predictions.append(np.full((x_tensor.shape[0], len(diag_cols)), np.inf, dtype=float))
            continue

        model.eval()
        with torch.no_grad():
            preds = torch.sigmoid(model(x_tensor)).cpu().numpy()
        all_predictions.append(preds)

    return all_predictions

def obs_nn_policy(x):
    all_predictions = obs_nn_predict_all_actions(x)
    action_scores = np.stack([preds.sum(axis=1) for preds in all_predictions], axis=0)
    return np.argmin(action_scores, axis=0)

print(
    {
        action_name: int(count)
        for action_name, count in zip(sglt_full[treatment_col].cat.categories, obs_nn_train_counts)
    }
)

{'No treatment': 5887, 'dapagliflozin': 163, 'empagliflozin': 375, 'ertugliflozin': 7}


In [13]:
DR_test_X_obs = scm.encode_data(DR_test_X[observation_cols], 'obs', fit=False)
DR_test_X_diags = torch.tensor(DR_test_X[diag_cols].astype(np.float32).to_numpy(), dtype=torch.float32, device=device)
policy_test_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(torch.cat([DR_test_X_obs.to(device), DR_test_X_diags.to(device)], dim=1), torch.zeros(len(DR_test_X)).to(device)),
    batch_size=32,
    shuffle=True
)

In [14]:
scm_policy_actions = best_action(DR_test_X, torch.tensor(DR_test_X.index), use_counterfactuals=True)

DR_test_X_combined = torch.cat([DR_test_X_obs.cpu(), DR_test_X_diags.cpu()], dim=1)

gbt_policy_actions = gbt_policy(DR_test_X_combined)
obs_gbt_policy_actions = obs_gbt_policy(DR_test_X_combined)
obs_nn_policy_actions = obs_nn_policy(DR_test_X_combined.to(device))
    
nn_policy_actions = policy(DR_test_X_combined.to(device))

# random policy baseline for sanity-check
rng = np.random.default_rng(42)
n_actions_eval = len(sglt_full[treatment_col].cat.categories)
random_policy_actions = rng.integers(0, n_actions_eval, size=len(DR_test_X))

In [15]:
def train_DR_model(train_data):
    has_followup_train = train_data['Next ESETAZON'].notna()
    train_indices = train_data[has_followup_train].index

    # train an outcome model for each diagnosis
    m_models = {dc: RandomForestClassifier(n_estimators=200, min_samples_leaf=5) for dc in diag_cols}
    m_train_X = train_data.loc[train_indices, observation_cols + diag_cols + [treatment_col]]

    m_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    m_encoder.fit(m_train_X)
    m_train_X = m_encoder.transform(m_train_X)
    m_train_Y = get_followups(train_data, train_data.loc[train_indices]['Next ESETAZON']).loc[:, diag_cols].astype(float).to_numpy()

    for dc in diag_cols:
        m_models[dc].fit(m_train_X, m_train_Y[:, diag_cols.index(dc)])

    # train a propensity model
    prop_model = LogisticRegression(max_iter=1000, multi_class='multinomial', penalty='l2', C=0.5, solver='lbfgs')
    p_train_X = train_data.loc[train_indices, observation_cols + diag_cols]
    p_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    p_encoder.fit(p_train_X)
    p_train_X = p_encoder.transform(p_train_X)
    p_train_Y = train_data.loc[train_indices][treatment_col].cat.codes.values
    prop_model.fit(p_train_X, p_train_Y)

    return m_models, m_encoder, prop_model, p_encoder

m_models, m_encoder, prop_model, p_encoder = train_DR_model(DR_train_X)

/home/sandordani/WORK/miniconda3/envs/work/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


In [17]:
# Compute DR estimates for all policies (NN, GBT, observational NN, SCM, action-only GBT, random)
has_followup_test = DR_test_X['Next ESETAZON'].notna()
test_indices = DR_test_X[has_followup_test].index

action_to_name = {i: name for i, name in enumerate(sglt_full[treatment_col].cat.categories)}
observed_actions = DR_test_X.loc[test_indices][treatment_col].cat.codes.values

gbt_best_actions = gbt_policy_actions[has_followup_test]
obs_gbt_best_actions = obs_gbt_policy_actions[has_followup_test]
obs_nn_best_actions = obs_nn_policy_actions[has_followup_test]
nn_best_actions = nn_policy_actions[has_followup_test]
scm_best_actions = scm_policy_actions[has_followup_test]
random_best_actions = random_policy_actions[has_followup_test]
true_actions = DR_test_X.loc[test_indices][treatment_col].cat.codes.values

true_outcomes = get_followups(DR_test_X, DR_test_X.loc[test_indices]['Next ESETAZON']).loc[:, diag_cols]

vobs = {}

def compute_dr_estimates_for_policy(policy_actions, test_data, outcome_model, outcome_encoder, prop_model, prop_encoder, only_eval_overlap=False, overlap_eps=0.05):
    action_df = pd.DataFrame([action_to_name[a] for a in policy_actions], columns=['TIPUS'])
    vcfs_policy = {}
    for dc in diag_cols:
        X_cf_policy = pd.concat([test_data.loc[test_indices, observation_cols + diag_cols].reset_index(drop=True), action_df], axis=1)
        X_cf_m_policy = outcome_encoder.transform(X_cf_policy)
        X_cf_p_policy = prop_encoder.transform(X_cf_policy[observation_cols + diag_cols])
        
        X_obs_m = outcome_encoder.transform(test_data.loc[test_indices, observation_cols + diag_cols + [treatment_col]])
        
        m_hat_cf_policy = outcome_model[dc].predict_proba(X_cf_m_policy)[:, 1]
        m_hat_obs = outcome_model[dc].predict_proba(X_obs_m)[:, 1]
        
        p_A_policy = prop_model.predict_proba(X_cf_p_policy)
        non_overlap_mask = np.any(p_A_policy < overlap_eps, axis=1) | np.any(p_A_policy > (1 - overlap_eps), axis=1)
        p_A_policy = p_A_policy[np.arange(len(p_A_policy)), policy_actions]
        # p_A_policy = np.clip(p_A_policy, 0.05, 0.95)
        indicator_policy = (observed_actions == policy_actions).astype(int)
        
        # DR estimate
        V_DR_cf_policy = m_hat_cf_policy + indicator_policy / p_A_policy * (true_outcomes[dc].values.astype(float) - m_hat_obs)
        V_DR_cf_policy = np.clip(V_DR_cf_policy, 0.0, 1.0)
        
        if only_eval_overlap:
            V_DR_cf_policy = V_DR_cf_policy[~non_overlap_mask]

        vcfs_policy[dc] = V_DR_cf_policy
    return vcfs_policy

def calculate_incidence_diffs(vcfs_policy, vobs):
    diffs = {}
    for dc in diag_cols:
        diffs[dc] = np.mean(vobs[dc]-vcfs_policy[dc])
    return diffs

vcfs_gbt = compute_dr_estimates_for_policy(gbt_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder)
vcfs_obs_gbt = compute_dr_estimates_for_policy(obs_gbt_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder)
vcfs_obs_nn = compute_dr_estimates_for_policy(obs_nn_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder)
vcfs_nn = compute_dr_estimates_for_policy(nn_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder)
vcfs_scm = compute_dr_estimates_for_policy(scm_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder)
vcfs_random = compute_dr_estimates_for_policy(random_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder)
vobs_DR = compute_dr_estimates_for_policy(true_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder)
vobs = {dc: true_outcomes[dc].values.astype(float) for dc in diag_cols}

diffs_gbt = calculate_incidence_diffs(vcfs_gbt, vobs_DR)
diffs_obs_gbt = calculate_incidence_diffs(vcfs_obs_gbt, vobs_DR)
diffs_obs_nn = calculate_incidence_diffs(vcfs_obs_nn, vobs_DR)
diffs_nn = calculate_incidence_diffs(vcfs_nn, vobs_DR)
diffs_scm = calculate_incidence_diffs(vcfs_scm, vobs_DR)
diffs_random = calculate_incidence_diffs(vcfs_random, vobs_DR)

# Compare results
for dc in diag_cols:
    print(f"\nDiagnosis: {dc}")
    print(f"GBT Policy DR estimate mean: {vcfs_gbt[dc].mean():.4f}")
    print(f"Observational GBT Policy DR estimate mean: {vcfs_obs_gbt[dc].mean():.4f}")
    print(f"Observational NN Policy DR estimate mean: {vcfs_obs_nn[dc].mean():.4f}")
    print(f"NN Policy DR estimate mean: {vcfs_nn[dc].mean():.4f}")
    print(f"SCM Policy DR estimate mean: {vcfs_scm[dc].mean():.4f}")
    print(f"Random Policy DR estimate mean: {vcfs_random[dc].mean():.4f}")
    print(f"Observed outcomes mean with DR: {vobs_DR[dc].mean():.4f}")
    print(f"Observed outcomes mean: {vobs[dc].mean():.4f}")


Diagnosis: Hepatopathia
GBT Policy DR estimate mean: 0.0758
Observational GBT Policy DR estimate mean: 0.0765
Observational NN Policy DR estimate mean: 0.0778
NN Policy DR estimate mean: 0.0749
SCM Policy DR estimate mean: 0.0505
Random Policy DR estimate mean: 0.0699
Observed outcomes mean with DR: 0.0830
Observed outcomes mean: 0.0830

Diagnosis: Fatty liver
GBT Policy DR estimate mean: 0.0788
Observational GBT Policy DR estimate mean: 0.0912
Observational NN Policy DR estimate mean: 0.1025
NN Policy DR estimate mean: 0.0848
SCM Policy DR estimate mean: 0.0648
Random Policy DR estimate mean: 0.1045
Observed outcomes mean with DR: 0.0859
Observed outcomes mean: 0.0859

Diagnosis: Diabetes
GBT Policy DR estimate mean: 0.5070
Observational GBT Policy DR estimate mean: 0.5147
Observational NN Policy DR estimate mean: 0.5256
NN Policy DR estimate mean: 0.5100
SCM Policy DR estimate mean: 0.4186
Random Policy DR estimate mean: 0.5251
Observed outcomes mean with DR: 0.5196
Observed outcome

In [18]:
# Optional: a more stable DR variant with propensity clipping.
def compute_dr_estimates_for_policy_stable(
    policy_actions,
    test_data,
    outcome_model,
    outcome_encoder,
    prop_model,
    prop_encoder,
    clip_eps=0.05,
):
    action_df = pd.DataFrame([action_to_name[a] for a in policy_actions], columns=['TIPUS'])
    vcfs_policy = {}
    for dc in diag_cols:
        X_cf_policy = pd.concat([
            test_data.loc[test_indices, observation_cols + diag_cols].reset_index(drop=True),
            action_df
        ], axis=1)
        X_cf_m_policy = outcome_encoder.transform(X_cf_policy)
        X_cf_p_policy = prop_encoder.transform(X_cf_policy[observation_cols + diag_cols])

        X_obs_m = outcome_encoder.transform(
            test_data.loc[test_indices, observation_cols + diag_cols + [treatment_col]]
        )

        m_hat_cf_policy = outcome_model[dc].predict_proba(X_cf_m_policy)[:, 1]
        m_hat_obs = outcome_model[dc].predict_proba(X_obs_m)[:, 1]

        p_A_policy = prop_model.predict_proba(X_cf_p_policy)
        p_A_policy = p_A_policy[np.arange(len(p_A_policy)), policy_actions]
        p_A_policy = np.clip(p_A_policy, clip_eps, 1.0 - clip_eps)

        indicator_policy = (observed_actions == policy_actions).astype(int)
        V_DR_cf_policy = m_hat_cf_policy + indicator_policy / p_A_policy * (
            true_outcomes[dc].values.astype(float) - m_hat_obs
        )
        vcfs_policy[dc] = np.clip(V_DR_cf_policy, 0.0, 1.0)

    return vcfs_policy

stable_vcfs = {
    'gbt': compute_dr_estimates_for_policy_stable(gbt_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder),
    'obs_gbt': compute_dr_estimates_for_policy_stable(obs_gbt_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder),
    'obs_nn': compute_dr_estimates_for_policy_stable(obs_nn_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder),
    'nn': compute_dr_estimates_for_policy_stable(nn_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder),
    'scm': compute_dr_estimates_for_policy_stable(scm_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder),
    'random': compute_dr_estimates_for_policy_stable(random_best_actions, DR_test_X, m_models, m_encoder, prop_model, p_encoder),
}

print("=== Mean incidence improvement (Observed raw outcomes - Policy DR, clipped) ===")
for model_name, vcfs in stable_vcfs.items():
    diffs = calculate_incidence_diffs(vcfs, vobs)
    print(f"{model_name:15s} mean_over_diseases={np.mean(list(diffs.values())):.5f}")

=== Mean incidence improvement (Observed raw outcomes - Policy DR, clipped) ===
gbt             mean_over_diseases=0.00679
obs_gbt         mean_over_diseases=0.00417
obs_nn          mean_over_diseases=0.00241
nn              mean_over_diseases=0.00423
scm             mean_over_diseases=0.03737
random          mean_over_diseases=0.00117


In [19]:
# Safer observational policy for stress-testing sparse-action artifacts.
# Keeps the same trained obs_gbt models but adds two guards:
# 1) clip predicted risks to [0, 1]
# 2) optionally disallow actions with too little follow-up support
action_names = list(sglt_full[treatment_col].cat.categories)
obs_train_df = policy_train_X.loc[cases_with_followup].copy()
obs_train_counts = obs_train_df[treatment_col].value_counts().reindex(action_names, fill_value=0)

min_support = 50
supported_action_idx = np.where(obs_train_counts.reindex(action_names, fill_value=0).values >= min_support)[0]


def obs_gbt_policy_safe(x, clip_outputs=True, disallow_sparse=True):
    x_np = x.cpu().numpy() if isinstance(x, torch.Tensor) else x
    action_scores = []

    for action_idx in range(n_actions_obs_gbt):
        preds = np.asarray(obs_gbt_models[action_idx].predict(x_np), dtype=float)
        if clip_outputs:
            preds = np.clip(preds, 0.0, 1.0)
        score = np.sum(preds, axis=1)

        if disallow_sparse and action_idx not in supported_action_idx:
            score = np.full_like(score, np.inf, dtype=float)

        action_scores.append(score)

    action_scores = np.stack(action_scores, axis=0)
    return np.argmin(action_scores, axis=0)

obs_gbt_safe_policy_actions = obs_gbt_policy_safe(DR_test_X_combined, clip_outputs=True, disallow_sparse=True)
obs_gbt_safe_best_actions = obs_gbt_safe_policy_actions[has_followup_test]

safe_counts = pd.Series(obs_gbt_safe_best_actions).value_counts().sort_index()
safe_counts.index = [action_names[i] for i in safe_counts.index]
safe_counts = safe_counts.reindex(action_names, fill_value=0)
print("=== obs_gbt_safe selected action shares (eval subset) ===")
print((safe_counts / max(safe_counts.sum(), 1)).rename('share'))

if 'compute_dr_estimates_for_policy_stable' in globals():
    vcfs_obs_gbt_safe = compute_dr_estimates_for_policy_stable(
        obs_gbt_safe_best_actions,
        DR_test_X,
        m_models,
        m_encoder,
        prop_model,
        p_encoder,
        clip_eps=0.05,
    )
    diffs_obs_gbt_safe = calculate_incidence_diffs(vcfs_obs_gbt_safe, vobs)
    print("\nMean incidence improvement (Observed raw outcomes - Policy DR, clipped):")
    print(f"obs_gbt original: {np.mean(list(calculate_incidence_diffs(stable_vcfs['obs_gbt'], vobs).values())):.5f}")
    print(f"obs_gbt safe    : {np.mean(list(diffs_obs_gbt_safe.values())):.5f}")
else:
    print("\nStable DR helper not found in memory. Run the stable DR cell first.")

=== obs_gbt_safe selected action shares (eval subset) ===
No treatment     0.520748
dapagliflozin    0.271186
empagliflozin    0.208065
ertugliflozin    0.000000
Name: share, dtype: float64

Mean incidence improvement (Observed raw outcomes - Policy DR, clipped):
obs_gbt original: 0.00417
obs_gbt safe    : 0.00368


In [20]:
# bootstrap DR estimates to get confidence intervals
n_bootstrap = 100

policy_actions_map = {
    'gbt': gbt_best_actions,
    'obs_gbt': obs_gbt_safe_best_actions,
    'obs_nn': obs_nn_best_actions,
    'nn': nn_best_actions,
    'scm': scm_best_actions,
    'random': random_best_actions,
}

policy_labels = {
    'gbt': 'GBT Policy',
    'obs_gbt': 'Obs GBT Policy',
    'obs_nn': 'Obs NN Policy',
    'action_only_gbt': 'Action-only GBT',
    'nn': 'NN Policy',
    'scm': 'SCM Policy',
    'random': 'Random Policy',
}

np.random.seed(42)


bootstrap_diffs = {model_name: {dc: [] for dc in diag_cols} for model_name in policy_actions_map}
bootstrap_outcomes = {model_name: {dc: [] for dc in diag_cols} for model_name in policy_actions_map}
bootstrap_true_outcomes = {dc: [] for dc in diag_cols}
bootstrap_daly_improvements = {model_name: [] for model_name in policy_actions_map}

bootsrap_test_data = DR_test_X.loc[test_indices].copy()
bootstrap_sexes = bootsrap_test_data['NEM'].values
bootstrap_ages = np.array([int(age_cats[a]) for a in bootsrap_test_data['ESETKOR']])

def outcomes_dict_to_matrix(outcomes_dict):
    return np.column_stack([outcomes_dict[dc] for dc in diag_cols])

# Patient-level bootstrap with replacement, then unique IDs to avoid duplicated
# ESETAZON collisions inside get_followups().
unique_patient_ids = DR_train_X['SZEMELYID'].unique()

for i in tqdm(range(n_bootstrap)):
    sampled_patient_ids = np.random.choice(unique_patient_ids, size=len(unique_patient_ids), replace=True)
    resampled_patient_ids = np.unique(sampled_patient_ids)
    train_data_sample = DR_train_X[DR_train_X['SZEMELYID'].isin(resampled_patient_ids)]
    m_models_sample, m_encoder_sample, prop_model_sample, p_encoder_sample = train_DR_model(train_data_sample)

    policy_vcfs = {
        model_name: compute_dr_estimates_for_policy_stable(actions, bootsrap_test_data, m_models_sample, m_encoder_sample, prop_model_sample, p_encoder_sample)
        for model_name, actions in policy_actions_map.items()
    }
    vobs_DR_sample = compute_dr_estimates_for_policy_stable(true_actions, bootsrap_test_data, m_models_sample, m_encoder_sample, prop_model_sample, p_encoder_sample)

    for model_name, outcomes_sample in policy_vcfs.items():
        diffs_sample = calculate_incidence_diffs(outcomes_sample, vobs_DR_sample)
        for dc in diag_cols:
            bootstrap_diffs[model_name][dc].append(diffs_sample[dc])
            bootstrap_outcomes[model_name][dc].append(outcomes_sample[dc])

    for dc in diag_cols:
        bootstrap_true_outcomes[dc].append(vobs_DR_sample[dc])

    vobs_dr_matrix_sample = outcomes_dict_to_matrix(vobs_DR_sample)
    vobs_daly_mean = daly(vobs_dr_matrix_sample, bootstrap_sexes, bootstrap_ages).sum(axis=1).mean()

    for model_name, outcomes_sample in policy_vcfs.items():
        model_matrix_sample = outcomes_dict_to_matrix(outcomes_sample)
        model_daly_mean = daly(model_matrix_sample, bootstrap_sexes, bootstrap_ages).sum(axis=1).mean()
        bootstrap_daly_improvements[model_name].append(vobs_daly_mean - model_daly_mean)
        # bootstrap_daly_improvements[model_name].append(model_daly_mean)

    print('gbt')
    gbt_model_matrix_sample = outcomes_dict_to_matrix(policy_vcfs['gbt'])
    gbt_dalys = daly(gbt_model_matrix_sample, bootstrap_sexes, bootstrap_ages).sum(axis=1)
    obs_gbt_model_matrix_sample = outcomes_dict_to_matrix(policy_vcfs['obs_gbt'])
    obs_gbt_dalys = daly(obs_gbt_model_matrix_sample, bootstrap_sexes, bootstrap_ages).sum(axis=1)
    bad_samples = obs_gbt_dalys < gbt_dalys
    good_samples = obs_gbt_dalys > gbt_dalys



policy_CIs = {
    model_name: {'lower': {}, 'upper': {}, 'mean': {}}
    for model_name in policy_actions_map
}

daly_CIs = {
    model_name: {
        'lower': np.percentile(values, 2.5),
        'mean': np.mean(values),
        'upper': np.percentile(values, 97.5)
    }
    for model_name, values in bootstrap_daly_improvements.items()
}

# Calculate confidence intervals
for dc in diag_cols:
    for model_name in policy_actions_map:
        ci = policy_CIs[model_name]
        ci['lower'][dc] = np.percentile(bootstrap_diffs[model_name][dc], 2.5)
        ci['mean'][dc] = np.mean(bootstrap_diffs[model_name][dc])
        ci['upper'][dc] = np.percentile(bootstrap_diffs[model_name][dc], 97.5)

# Backward-compatible aliases used by downstream cells
bootstrap_diffs_gbt = bootstrap_diffs['gbt']
bootstrap_diffs_obs_gbt = bootstrap_diffs['obs_gbt']
bootstrap_diffs_obs_nn = bootstrap_diffs['obs_nn']
bootstrap_diffs_action_only_gbt = bootstrap_diffs['action_only_gbt']
bootstrap_diffs_nn = bootstrap_diffs['nn']
bootstrap_diffs_scm = bootstrap_diffs['scm']
bootstrap_diffs_random = bootstrap_diffs['random']

bootstrap_gbt_outcomes = bootstrap_outcomes['gbt']
bootstrap_obs_gbt_outcomes = bootstrap_outcomes['obs_gbt']
bootstrap_obs_nn_outcomes = bootstrap_outcomes['obs_nn']
bootstrap_action_only_gbt_outcomes = bootstrap_outcomes['action_only_gbt']
bootstrap_nn_outcomes = bootstrap_outcomes['nn']
bootstrap_scm_outcomes = bootstrap_outcomes['scm']
bootstrap_random_outcomes = bootstrap_outcomes['random']

gbt_CIs = policy_CIs['gbt']
obs_gbt_CIs = policy_CIs['obs_gbt']
obs_nn_CIs = policy_CIs['obs_nn']
action_only_gbt_CIs = policy_CIs['action_only_gbt']
nn_CIs = policy_CIs['nn']
scm_CIs = policy_CIs['scm']
random_CIs = policy_CIs['random']

for dc in diag_cols:
    print(f"\nDiagnosis: {dc}")
    for model_name in policy_actions_map:
        ci = policy_CIs[model_name]
        print(
            f"{policy_labels[model_name]} DR estimate mean: {ci['mean'][dc]:.4f} "
            f"(95% CI: [{ci['lower'][dc]:.4f}, {ci['upper'][dc]:.4f}])"
        )

print("\nBootstrap DALY improvement (Observed - Policy; positive = lower DALY under policy):")
for model_name, stats in daly_CIs.items():
    print(
        f"{model_name}: {stats['mean']:.6f} "
        f"(95% CI: [{stats['lower']:.6f}, {stats['upper']:.6f}])"
    )

plt.figure(figsize=(14, 6))
x = np.arange(len(diag_cols))
width = 0.12
model_order = ['gbt', 'obs_gbt', 'obs_nn', 'nn', 'scm', 'random']
offsets = np.array([-3, -2, -1, 0, 1, 2]) * width

for model_name, offset in zip(model_order, offsets):
    ci = policy_CIs[model_name]
    plt.bar(
        x + offset,
        [ci['mean'][dc] for dc in diag_cols],
        width=width,
        label=policy_labels[model_name],
        yerr=[(ci['upper'][dc] - ci['lower'][dc]) / 2 for dc in diag_cols],
        capsize=5
    )

plt.xticks(x, diag_cols, rotation=45)
plt.ylabel('Estimated Incidence Difference (Policy vs Observed)')
plt.title('Estimated Incidence Differences with 95% Confidence Intervals')
plt.legend()

  0%|          | 0/100 [00:00<?, ?it/s]/home/sandordani/WORK/miniconda3/envs/work/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
  1%|          | 1/100 [00:19<32:45, 19.85s/it]

gbt


/home/sandordani/WORK/miniconda3/envs/work/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
  2%|▏         | 2/100 [00:40<33:23, 20.45s/it]

gbt


/home/sandordani/WORK/miniconda3/envs/work/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
  3%|▎         | 3/100 [01:00<32:56, 20.37s/it]

...

/home/sandordani/WORK/miniconda3/envs/work/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
 99%|█████████▉| 99/100 [34:04<00:20, 20.81s/it]

gbt


/home/sandordani/WORK/miniconda3/envs/work/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
100%|██████████| 100/100 [34:24<00:00, 20.65s/it]

gbt


KeyError: 'action_only_gbt'

In [ ]:
# Plot DALY improvements with 95% confidence intervals
model_order = ['gbt', 'nn', 'scm']
model_labels = ['GBT Policy', 'NN Policy', 'SCM Policy']

daly_means = [daly_CIs[m]['mean'] for m in model_order]
daly_yerrs = [(daly_CIs[m]['upper'] - daly_CIs[m]['lower']) / 2 for m in model_order]

plt.figure(figsize=(10, 5))
x = np.arange(len(model_order))
plt.bar(x, daly_means, yerr=daly_yerrs, capsize=5)
plt.xticks(x, model_labels, rotation=20, ha='right')
plt.ylabel('Estimated DALY Difference (Observed - Policy)')
plt.title('Estimated DALY Differences with 95% Confidence Intervals')
plt.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# plot frequency of missed actions for doctors
action_conf_matrix_real = np.zeros((len(sglt_full[treatment_col].cat.categories), len(sglt_full[treatment_col].cat.categories)), dtype=int)


for i in range(len(sglt_full[treatment_col].cat.categories)):
    for j in range(len(sglt_full[treatment_col].cat.categories)):
        action_conf_matrix_real[i, j] = np.sum((observed_actions == i) & (scm_best_actions == j))


cmap = LinearSegmentedColormap.from_list(
    "custom",
    ["#ffffff", "#fc9e4f"] 
)


sns.heatmap(action_conf_matrix_real, annot=True, fmt='d', cmap=cmap, xticklabels=sglt_full[treatment_col].cat.categories, yticklabels=sglt_full[treatment_col].cat.categories)
plt.title('Observed Actions vs SCM Counterfactual Optimal Actions')
plt.xlabel('Policy Actions')
plt.ylabel('Observed Actions')
plt.show()



In [ ]:
plt.bar(vobs.keys(), [np.mean(vobs[dc]) for dc in vobs.keys()], color='#fc9e4f')
plt.title('Observed incidence rates')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Diseases')
plt.ylabel('Incidence\n (lower is better)')

In [ ]:
plt.bar(diffs_nn.keys(), diffs_nn.values(), color='#fc9e4f')
plt.title('Improvement of outcomes of SCM based treatment policy (MLP)\n over observed outcomes based on DR estimates')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Diseases')
plt.ylabel('Difference in incidence\n (higher is better)')

In [ ]:
plt.bar(diffs_gbt.keys(), diffs_gbt.values(), color='#fc9e4f')
plt.title('Improvement of outcomes of SCM based treatment policy (GBT)\n over observed outcomes based on DR estimates')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Diseases')
plt.ylabel('Difference in incidence\n (higher is better)')

In [ ]:
plt.bar(diffs_scm.keys(), diffs_scm.values(), color='#fc9e4f')
plt.title('Improvement of outcomes of SCM based treatment policy (SCM)\n over observed outcomes based on DR estimates')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Diseases')
plt.ylabel('Difference in incidence\n (higher is better)')

In [ ]:
# Plot comparison of NN vs GBT improvements
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot comparison
x_pos = np.arange(len(diag_cols))
width = 0.35

axes[0].bar(x_pos - width/2, [diffs_nn[dc] for dc in diag_cols], width, 
            label='Neural Network', color='#fc9e4f', alpha=0.8)
axes[0].bar(x_pos + width/2, [diffs_gbt[dc] for dc in diag_cols], width, 
            label='GBT', color='#7daa92', alpha=0.8)

axes[0].set_xlabel('Diseases')
axes[0].set_ylabel('Risk Improvement')
axes[0].set_title('Policy Improvement: NN vs GBT')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(diag_cols, rotation=45, ha='right')
axes[0].legend()
axes[0].axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
axes[0].grid(axis='y', alpha=0.3)

# Scatter plot: GBT vs NN improvements
nn_improvements = [diffs_nn[dc] for dc in diag_cols]
gbt_improvements = [diffs_gbt[dc] for dc in diag_cols]

axes[1].scatter(nn_improvements, gbt_improvements, s=100, alpha=0.7, color='#c44569')
axes[1].plot([min(nn_improvements + gbt_improvements), max(nn_improvements + gbt_improvements)], 
             [min(nn_improvements + gbt_improvements), max(nn_improvements + gbt_improvements)], 
             'k--', linewidth=1, label='Equal performance')

for i, dc in enumerate(diag_cols):
    axes[1].annotate(dc, (nn_improvements[i], gbt_improvements[i]), 
                    fontsize=8, alpha=0.7, xytext=(5, 5), 
                    textcoords='offset points')

axes[1].set_xlabel('NN Policy Improvement')
axes[1].set_ylabel('GBT Policy Improvement')
axes[1].set_title('Policy Improvement: GBT vs NN\n(Points above line = GBT better)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

np.random.seed(42)
n = 120
X = torch.cat([scm.encode_data(DR_test_X.loc[test_indices][observation_cols], 'obs', fit=False).cpu(), torch.tensor(DR_test_X.loc[test_indices][diag_cols].astype(np.float32).to_numpy(), dtype=torch.float32)], dim=1).numpy()

doctor_labels = observed_actions

X_pca = PCA(n_components=20).fit_transform(X)
tsne = TSNE(n_components=2, perplexity=20, learning_rate='auto')
X_emb = tsne.fit_transform(X_pca)

plt.figure(figsize=(7,5))
plt.scatter(X_emb[:,0], X_emb[:,1], c=doctor_labels)
plt.title("UMAP-style Embedding (t-SNE fallback) Colored by Doctor Treatment")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.show()

plt.figure(figsize=(7,5))
plt.scatter(X_emb[:,0], X_emb[:,1], c=scm_best_actions)
plt.title("UMAP-style Embedding (t-SNE fallback) Colored by MLP Treatment")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.show()

plt.figure(figsize=(7,5))
plt.scatter(X_emb[:,0], X_emb[:,1], c=gbt_best_actions)
plt.title("UMAP-style Embedding (t-SNE fallback) Colored by GBT Treatment")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import mutual_info_score

doctor_mis = {}
scm_mis = {}

for obs_col in DR_test_X.loc[test_indices][observation_cols].columns:
    mask = DR_test_X.loc[test_indices][obs_col].notna() * (observed_actions != scm_best_actions)
    if mask.sum() == 0:
        continue
    mi_doctor = mutual_info_score(DR_test_X.loc[test_indices][obs_col][mask], doctor_labels[mask])
    mi_scm = mutual_info_score(DR_test_X.loc[test_indices][obs_col][mask], scm_best_actions[mask])
    doctor_mis[obs_col] = mi_doctor
    scm_mis[obs_col] = mi_scm
    # print(f"Mutual Information between {obs_col} and Doctor Actions: {mi_doctor}, SCM Actions: {mi_scm}")

scm_mis['AGE'] = scm_mis.pop('ESETKOR')
doctor_mis['AGE'] = doctor_mis.pop('ESETKOR')
scm_mis['SEX'] = scm_mis.pop('NEM')
doctor_mis['SEX'] = doctor_mis.pop('NEM')



plt.figure(figsize=(6,6))
width = 0.4
x = np.arange(len(doctor_mis.keys()))
doctor_mi_values = [doctor_mis[obs_col] for obs_col in doctor_mis.keys()]
scm_mi_values = [scm_mis[obs_col] for obs_col in scm_mis.keys()]
plt.bar(x - width/2, doctor_mi_values, width, label='Observed Treatments', color='#576574')
plt.bar(x + width/2, scm_mi_values, width, label='SCM Treatments', color='#c44569')
plt.xticks(x, doctor_mis.keys(), rotation=45, ha='right')
plt.ylabel('Mutual Information')
plt.title('Mutual Information between Observations and Actions\n For Cases with Disagreement')
plt.legend()
plt.show()

In [ ]:
new_sglt_patients = DR_test_X.loc[test_indices][(observed_actions == 0) & (scm_best_actions != 0)]


np.random.seed(42)
n = 120
X = torch.cat([scm.encode_data(DR_test_X.loc[test_indices][observation_cols], 'obs', fit=False).cpu(), torch.tensor(DR_test_X.loc[test_indices][diag_cols].astype(np.float32).to_numpy(), dtype=torch.float32)], dim=1).numpy()


X_pca = PCA(n_components=20).fit_transform(X)
tsne = TSNE(n_components=2, perplexity=20, learning_rate='auto')
X_emb = tsne.fit_transform(X_pca)

plt.figure(figsize=(7,5))
plt.scatter(X_emb[:,0], X_emb[:,1], c=(observed_actions == 0) & (scm_best_actions != 0))
plt.title("UMAP-style Embedding (t-SNE fallback) New SGLT2 treatments recommended by SCM but not by doctors")
plt.xlabel("Dim 1")
plt.ylabel("Dim 2")
plt.legend()
plt.show()
